# Practical 5 — Leakage-Aware Feature Engineering

## Objective

Transform validated log records into reproducible machine-learning features while preserving event identity and preventing information leakage.

The feature design separates:

1. **Identifiers** — retained for traceability, never supplied to the model.
2. **Event-local features** — derived only from the current event.
3. **Past-only behavioural features** — derived from events previously observed for the client.
4. **Restricted labeling features** — excluded from the primary model because they generated the weak labels.
5. **Target metadata** — labels and confidence information, never treated as input features.

This separation prevents the classifier from merely copying the weak-label rules.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


def locate_project_root(start_path):
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = locate_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.provenance import calculate_file_sha256


CLEANED_PATH = (
    PROJECT_ROOT / "data" / "processed" / "cj_cleaned.csv"
)

CLEANING_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cj_cleaning_manifest.json"
)

LABELS_PATH = (
    PROJECT_ROOT / "data" / "labels" / "cj_weak_labels.csv"
)

LABELS_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "labels"
    / "cj_weak_labels_manifest.json"
)

CHUNK_SIZE = 100_000
NULL_SENTINEL = r"\N"

with CLEANING_MANIFEST_PATH.open("r", encoding="utf-8") as file:
    cleaning_manifest = json.load(file)

with LABELS_MANIFEST_PATH.open("r", encoding="utf-8") as file:
    labels_manifest = json.load(file)

print("Project root:", PROJECT_ROOT)
print("Cleaned records:", f"{cleaning_manifest['output_rows']:,}")
print("Label records:", f"{labels_manifest['output_rows']:,}")
print("Label version:", labels_manifest["label_version"])

Project root: C:\Users\diyas\Desktop\PDS-Log-IDS-Project
Cleaned records: 2,062,361
Label records: 2,062,361
Label version: weak-label-v1


## Input contract verification

Feature engineering must operate on the exact cleaned dataset used to generate the weak labels. Fingerprints, row counts, and event identities are checked before deriving any features.

In [2]:
actual_cleaned_hash = calculate_file_sha256(CLEANED_PATH)
actual_labels_hash = calculate_file_sha256(LABELS_PATH)

assert (
    actual_cleaned_hash
    == cleaning_manifest["output_sha256"]
), "Cleaned dataset fingerprint mismatch."

assert (
    actual_cleaned_hash
    == labels_manifest["input_sha256"]
), "Labels were generated from a different cleaned dataset."

assert (
    actual_labels_hash
    == labels_manifest["output_sha256"]
), "Weak-label sidecar fingerprint mismatch."

assert (
    cleaning_manifest["output_rows"]
    == labels_manifest["input_rows"]
    == labels_manifest["output_rows"]
), "Input and label row counts differ."

cleaned_identity_sample = pd.read_csv(
    CLEANED_PATH,
    usecols=["event_id", "record_hash"],
    dtype=str,
    keep_default_na=False,
    nrows=5,
)

label_identity_sample = pd.read_csv(
    LABELS_PATH,
    usecols=["event_id", "record_hash"],
    dtype=str,
    keep_default_na=False,
    nrows=5,
)

assert cleaned_identity_sample.equals(
    label_identity_sample
), "Initial event identities are misaligned."

print("Cleaned fingerprint: passed")
print("Label fingerprint: passed")
print("Row-count contract: passed")
print("Initial identity alignment: passed")

Cleaned fingerprint: passed
Label fingerprint: passed
Row-count contract: passed
Initial identity alignment: passed


## Feature contract

Each column receives an explicit role:

- **Identity:** retained only to join artifacts.
- **Provenance:** retained for auditing, never modeled.
- **Grouping-only:** used to maintain behavioural state, never modeled directly.
- **Allowed source:** may generate primary-model features.
- **Restricted source:** participated in weak labeling and is excluded from the primary model.
- **Target metadata:** describes the weak target and is never an input feature.

`client_ip_canonical` is grouping-only. Supplying it directly to a model would encourage memorization of known clients rather than learning transferable behaviour.

In [3]:
from src.labeling import (
    EVIDENCE_SOURCE_COLUMNS,
    LABEL_COLUMNS,
)

IDENTITY_COLUMNS = {
    "event_id",
    "record_hash",
}

PROVENANCE_COLUMNS = {
    "source_line",
    "array_position",
}

GROUPING_ONLY_COLUMNS = {
    "client_ip",
    "client_ip_canonical",
}

PRIMARY_ALLOWED_SOURCES = {
    "timestamp_normalized",
    "source_port_normalized",
}

RESTRICTED_LABEL_SOURCES = set(EVIDENCE_SOURCE_COLUMNS)

TARGET_METADATA_COLUMNS = set(LABEL_COLUMNS)

assert PRIMARY_ALLOWED_SOURCES.isdisjoint(
    RESTRICTED_LABEL_SOURCES
), "An allowed source also participates in weak labeling."

assert GROUPING_ONLY_COLUMNS.isdisjoint(
    PRIMARY_ALLOWED_SOURCES
), "A grouping identifier was incorrectly approved as a feature."

assert TARGET_METADATA_COLUMNS.isdisjoint(
    PRIMARY_ALLOWED_SOURCES
), "Target metadata was incorrectly approved as a feature."

cleaned_columns = set(
    pd.read_csv(CLEANED_PATH, nrows=0).columns
)

label_columns = set(
    pd.read_csv(LABELS_PATH, nrows=0).columns
)

required_cleaned_columns = (
    IDENTITY_COLUMNS
    | PROVENANCE_COLUMNS
    | GROUPING_ONLY_COLUMNS
    | PRIMARY_ALLOWED_SOURCES
    | RESTRICTED_LABEL_SOURCES
)

missing_cleaned_columns = (
    required_cleaned_columns - cleaned_columns
)

missing_label_columns = (
    IDENTITY_COLUMNS
    | TARGET_METADATA_COLUMNS
) - label_columns

if missing_cleaned_columns:
    raise RuntimeError(
        f"Missing cleaned columns: {sorted(missing_cleaned_columns)}"
    )

if missing_label_columns:
    raise RuntimeError(
        f"Missing label columns: {sorted(missing_label_columns)}"
    )

source_contract = pd.DataFrame(
    [
        {
            "role": "identity",
            "columns": sorted(IDENTITY_COLUMNS),
            "model_input": False,
        },
        {
            "role": "provenance",
            "columns": sorted(PROVENANCE_COLUMNS),
            "model_input": False,
        },
        {
            "role": "grouping_only",
            "columns": sorted(GROUPING_ONLY_COLUMNS),
            "model_input": False,
        },
        {
            "role": "primary_allowed_source",
            "columns": sorted(PRIMARY_ALLOWED_SOURCES),
            "model_input": True,
        },
        {
            "role": "restricted_label_source",
            "columns": sorted(RESTRICTED_LABEL_SOURCES),
            "model_input": False,
        },
        {
            "role": "target_metadata",
            "columns": sorted(TARGET_METADATA_COLUMNS),
            "model_input": False,
        },
    ]
)

print("Feature-source contract: passed")
print(
    "Restricted weak-label sources:",
    sorted(RESTRICTED_LABEL_SOURCES),
)

source_contract

Feature-source contract: passed
Restricted weak-label sources: ['category_type', 'language', 'metadata', 'sub_key', 'user_agent']


,role,columns,model_input
0,identity,"[event_id, record_hash]",False
1,provenance,"[array_position, source_line]",False
2,grouping_only,"[client_ip, client_ip_canonical]",False
3,primary_allowed_source,"[source_port_normalized, timestamp_normalized]",True
4,restricted_label_source,"[category_type, language, metadata, sub_key, u...",False
5,target_metadata,"[evidence_codes, evidence_count, label_confide...",False


## Planned primary features

### Event-local features

- cyclical hour representation;
- cyclical weekday representation;
- weekend indicator;
- numerical source port;
- privileged, registered, and dynamic port indicators;
- language presence and length.

### Past-only client behaviour

- number of previously observed events;
- time since the previous client event;
- timestamp-reversal indicator;
- whether the source port matches the previous event;
- whether the port was previously observed;
- number of previously observed unique ports;
- exponentially decayed activity over multiple time scales.

For decay window \(\tau\), the activity before event \(i\) is:
For decay window $\tau$, the activity before event $i$ is:

$$
A_{\tau}(t_i)
=
\sum_{j<i}
e^{-\max(0,\,t_i-t_j)/\tau}
$$

Only earlier source-order events contribute. This provides bounded-memory burst information without looking into future records.

In [4]:
feature_registry = pd.DataFrame(
    [
        ("hour_sin", "event_local", "timestamp_normalized"),
        ("hour_cos", "event_local", "timestamp_normalized"),
        ("weekday_sin", "event_local", "timestamp_normalized"),
        ("weekday_cos", "event_local", "timestamp_normalized"),
        ("is_weekend", "event_local", "timestamp_normalized"),
        ("source_port_value", "event_local", "source_port_normalized"),
        ("is_privileged_port", "event_local", "source_port_normalized"),
        ("is_registered_port", "event_local", "source_port_normalized"),
        ("is_dynamic_port", "event_local", "source_port_normalized"),
        ("language_present", "event_local", "language"),
        ("language_length", "event_local", "language"),
        (
            "client_prior_event_count_log1p",
            "past_only",
            "client_ip_canonical",
        ),
        (
        "has_previous_event",
        "past_only",
        "client_ip_canonical",
        ),
        (
            "seconds_since_previous_log1p",
            "past_only",
            "client_ip_canonical + timestamp_normalized",
        ),
        (
            "timestamp_reversal_flag",
            "past_only",
            "client_ip_canonical + timestamp_normalized",
        ),
        (
            "same_port_as_previous",
            "past_only",
            "client_ip_canonical + source_port_normalized",
        ),
        (
            "source_port_seen_before",
            "past_only",
            "client_ip_canonical + source_port_normalized",
        ),
        (
            "prior_unique_ports_log1p",
            "past_only",
            "client_ip_canonical + source_port_normalized",
        ),
        (
            "activity_decay_60s",
            "past_only",
            "client_ip_canonical + timestamp_normalized",
        ),
        (
            "activity_decay_300s",
            "past_only",
            "client_ip_canonical + timestamp_normalized",
        ),
        (
            "activity_decay_3600s",
            "past_only",
            "client_ip_canonical + timestamp_normalized",
        ),
    ],
    columns=[
        "feature",
        "feature_family",
        "source",
    ],
)

assert feature_registry["feature"].is_unique
assert set(feature_registry["feature"]).isdisjoint(
    TARGET_METADATA_COLUMNS
)

print("Registered features:", len(feature_registry))
print(
    feature_registry["feature_family"]
    .value_counts()
    .rename("feature_count")
)

feature_registry

Registered features: 21
feature_family
event_local    11
past_only      10
Name: feature_count, dtype: int64


,feature,feature_family,source
0,hour_sin,event_local,timestamp_normalized
1,hour_cos,event_local,timestamp_normalized
2,weekday_sin,event_local,timestamp_normalized
3,weekday_cos,event_local,timestamp_normalized
4,is_weekend,event_local,timestamp_normalized
5,source_port_value,event_local,source_port_normalized
6,is_privileged_port,event_local,source_port_normalized
7,is_registered_port,event_local,source_port_normalized
8,is_dynamic_port,event_local,source_port_normalized
9,language_present,event_local,language


## Event-local transformation tests

Before processing the complete dataset, boundary values and invariants are tested with synthetic records. This is preferable to testing only ordinary examples because most transformation bugs occur at boundaries.

In [5]:
from pathlib import Path
import sys

def locate_project_root(start_path):
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the project root.")

PROJECT_ROOT = locate_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\diyas\Desktop\PDS-Log-IDS-Project


In [6]:
from src.features import (
    EVENT_LOCAL_FEATURES,
    FEATURE_VERSION,
    build_event_local_features,
)

synthetic_events = pd.DataFrame(
    {
        "timestamp_normalized": [
            "2024-01-01 00:00:00",
            "2024-01-01 12:00:00",
            "2024-01-05 23:59:59",
            "2024-01-06 00:00:00",
        ],
        "source_port_normalized": [
            80,
            1024,
            49151,
            49152,
        ],
        "language": [
            NULL_SENTINEL,
            "en",
            "",
            None,
        ],
    }
)

original_synthetic_events = synthetic_events.copy(deep=True)

event_local_result = build_event_local_features(
    synthetic_events
)

assert synthetic_events.equals(
    original_synthetic_events
), "The source dataframe was modified."

assert list(event_local_result.columns) == EVENT_LOCAL_FEATURES

assert event_local_result["is_privileged_port"].tolist() == [
    1, 0, 0, 0
]

assert event_local_result["is_registered_port"].tolist() == [
    0, 1, 1, 0
]

assert event_local_result["is_dynamic_port"].tolist() == [
    0, 0, 0, 1
]

assert event_local_result["language_present"].tolist() == [
    0, 1, 0, 0
]

assert event_local_result["language_length"].tolist() == [
    0, 2, 0, 0
]

assert event_local_result["is_weekend"].tolist() == [
    0, 0, 0, 1
]

hour_radius = (
    event_local_result["hour_sin"] ** 2
    + event_local_result["hour_cos"] ** 2
)

weekday_radius = (
    event_local_result["weekday_sin"] ** 2
    + event_local_result["weekday_cos"] ** 2
)

assert np.allclose(hour_radius, 1.0, atol=1e-6)
assert np.allclose(weekday_radius, 1.0, atol=1e-6)

second_result = build_event_local_features(
    synthetic_events
)

pd.testing.assert_frame_equal(
    event_local_result,
    second_result,
)

print("Source preservation: passed")
print("Port-boundary tests: passed")
print("Missing-language handling: passed")
print("Cyclical invariants: passed")
print("Transformation idempotence: passed")

event_local_result

Source preservation: passed
Port-boundary tests: passed
Missing-language handling: passed
Cyclical invariants: passed
Transformation idempotence: passed


,hour_sin,hour_cos,weekday_sin,weekday_cos,is_weekend,source_port_value,is_privileged_port,is_registered_port,is_dynamic_port,language_present,language_length
0,0.000000e+00,1.0,0.000000,1.000000,0,80,1,0,0,0,0
1,1.224647e-16,-1.0,0.433884,0.900969,0,1024,0,1,0,1,2
2,-7.272205e-05,1.0,-0.974926,-0.222531,0,49151,0,1,0,0,0
3,0.000000e+00,1.0,-0.974928,-0.222521,1,49152,0,0,1,0,0


## Causal client-state tests

The transformer is tested for:

- first-event behaviour;
- exact decay calculations;
- timestamp reversals;
- future independence;
- chunk-boundary invariance.

Chunk-boundary invariance means the result must not change when the same stream is divided into different CSV chunk sizes.

In [7]:
import numpy as np
import pandas as pd
import importlib

import src.features as feature_module

importlib.reload(feature_module)

from src.features import (
    ALL_PRIMARY_FEATURES,
    CausalClientFeatureTransformer,
    PAST_ONLY_FEATURES,
)
synthetic_stream = pd.DataFrame(
    {
        "client_ip_canonical": [
            "192.0.2.1",
            "192.0.2.1",
            "198.51.100.2",
            "192.0.2.1",
            "192.0.2.1",
        ],
        "timestamp_normalized": [
            "2024-01-01 00:00:00",
            "2024-01-01 00:01:00",
            "2024-01-01 00:00:15",
            "2024-01-01 00:00:30",
            "2024-01-01 00:02:00",
        ],
        "source_port_normalized": [
            80,
            80,
            443,
            80,
            443,
        ],
    }
)

whole_transformer = CausalClientFeatureTransformer()

whole_result = whole_transformer.transform_chunk(
    synthetic_stream
)

chunked_transformer = CausalClientFeatureTransformer()

chunked_result = pd.concat(
    [
        chunked_transformer.transform_chunk(
            synthetic_stream.iloc[:2]
        ),
        chunked_transformer.transform_chunk(
            synthetic_stream.iloc[2:]
        ),
    ]
)

pd.testing.assert_frame_equal(
    whole_result,
    chunked_result,
)

prefix_transformer = CausalClientFeatureTransformer()

prefix_result = prefix_transformer.transform_chunk(
    synthetic_stream.iloc[:3]
)

pd.testing.assert_frame_equal(
    prefix_result.reset_index(drop=True),
    whole_result.iloc[:3].reset_index(drop=True),
)

assert whole_result["has_previous_event"].tolist() == [
    0, 1, 0, 1, 1
]
print(
    whole_result[
        [
            "has_previous_event",
            "timestamp_reversal_flag",
            "seconds_since_previous_log1p",
        ]
    ]
)
assert whole_result["timestamp_reversal_flag"].tolist() == [
    0, 0, 0, 1, 0
]

assert whole_result["same_port_as_previous"].tolist() == [
    0, 1, 0, 1, 0
]

assert whole_result["source_port_seen_before"].tolist() == [
    0, 1, 0, 1, 0
]

assert np.isclose(
    whole_result.loc[1, "activity_decay_60s"],
    np.exp(-1),
    atol=1e-6,
)

assert whole_transformer.state_summary() == {
    "client_count": 2,
    "processed_events": 5,
    "client_port_pairs": 3,
}

assert len(ALL_PRIMARY_FEATURES) == 21
assert len(PAST_ONLY_FEATURES) == 10

print("First-event behaviour: passed")
print("Decay calculation: passed")
print("Timestamp-reversal handling: passed")
print("Future independence: passed")
print("Chunk-boundary invariance: passed")
print("State summary:", whole_transformer.state_summary())

whole_result

   has_previous_event  timestamp_reversal_flag  seconds_since_previous_log1p
0                   0                        0                      0.000000
1                   1                        0                      4.110874
2                   0                        0                      0.000000
3                   1                        1                      0.000000
4                   1                        0                      4.510859
First-event behaviour: passed
Decay calculation: passed
Timestamp-reversal handling: passed
Future independence: passed
Chunk-boundary invariance: passed
State summary: {'client_count': 2, 'processed_events': 5, 'client_port_pairs': 3}


,client_prior_event_count_log1p,has_previous_event,seconds_since_previous_log1p,timestamp_reversal_flag,same_port_as_previous,source_port_seen_before,prior_unique_ports_log1p,activity_decay_60s,activity_decay_300s,activity_decay_3600s
0,0.000000,0,0.000000,0,0,0,0.000000,0.000000,0.000000,0.000000
1,0.693147,1,4.110874,0,1,1,0.693147,0.367879,0.818731,0.983471
2,0.000000,0,0.000000,0,0,0,0.000000,0.000000,0.000000,0.000000
3,1.098612,1,0.000000,1,1,1,0.693147,1.367879,1.818731,1.983472
4,1.386294,1,4.510859,0,0,0,0.693147,0.871094,2.307781,2.934159


## Full-data preflight

The first contiguous chunk is used instead of random rows because causal features require the original arrival sequence. Random sampling would remove earlier client history and produce artificial time gaps and activity values.

In [8]:
from src.features import (
    ALL_PRIMARY_FEATURES,
    CausalClientFeatureTransformer,
    EVENT_LOCAL_FEATURES,
    PAST_ONLY_FEATURES,
    build_event_local_features,
)

PREFLIGHT_ROWS = 100_000

cleaned_preflight = pd.read_csv(
    CLEANED_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "client_ip_canonical",
        "timestamp_normalized",
        "source_port_normalized",
        "language",
    ],
    dtype={
        "event_id": str,
        "record_hash": str,
        "client_ip_canonical": str,
    },
    keep_default_na=False,
    nrows=PREFLIGHT_ROWS,
)

labels_preflight = pd.read_csv(
    LABELS_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "weak_label",
        "label_confidence",
    ],
    dtype={
        "event_id": str,
        "record_hash": str,
        "weak_label": str,
    },
    keep_default_na=False,
    nrows=PREFLIGHT_ROWS,
)

pd.testing.assert_frame_equal(
    cleaned_preflight[
        ["event_id", "record_hash"]
    ].reset_index(drop=True),
    labels_preflight[
        ["event_id", "record_hash"]
    ].reset_index(drop=True),
)

event_local_preflight = build_event_local_features(
    cleaned_preflight
)

preflight_transformer = CausalClientFeatureTransformer()

past_only_preflight = preflight_transformer.transform_chunk(
    cleaned_preflight
)

primary_preflight = pd.concat(
    [
        event_local_preflight,
        past_only_preflight,
    ],
    axis=1,
)

assert list(primary_preflight.columns) == ALL_PRIMARY_FEATURES

assert (
    feature_registry["feature"].tolist()
    == ALL_PRIMARY_FEATURES
), "Notebook registry and implementation differ."

forbidden_model_inputs = (
    IDENTITY_COLUMNS
    | PROVENANCE_COLUMNS
    | GROUPING_ONLY_COLUMNS
    | RESTRICTED_LABEL_SOURCES
    | TARGET_METADATA_COLUMNS
)

assert set(primary_preflight.columns).isdisjoint(
    forbidden_model_inputs
), "A forbidden column entered the primary feature matrix."

assert not primary_preflight.isna().any().any()

assert np.isfinite(
    primary_preflight.to_numpy(dtype=np.float64)
).all()

unique_counts = primary_preflight.nunique(
    dropna=False
).sort_values()

constant_features = unique_counts[
    unique_counts <= 1
].index.tolist()

print("Preflight rows:", f"{len(primary_preflight):,}")
print("Primary features:", len(primary_preflight.columns))
print("Identity alignment: passed")
print("Feature-registry agreement: passed")
print("Forbidden-input exclusion: passed")
print("Finite-value validation: passed")
print("Constant features in preflight:", constant_features)
print("State summary:", preflight_transformer.state_summary())
print(
    "Timestamp reversals in preflight:",
    int(
        primary_preflight[
            "timestamp_reversal_flag"
        ].sum()
    ),
)

primary_preflight.describe().T

Preflight rows: 100,000
Primary features: 21
Identity alignment: passed
Feature-registry agreement: passed
Forbidden-input exclusion: passed
Finite-value validation: passed
Constant features in preflight: ['is_privileged_port']
State summary: {'client_count': 7997, 'processed_events': 100000, 'client_port_pairs': 65196}
Timestamp reversals in preflight: 1


,count,mean,std,min,25%,50%,75%,max
hour_sin,100000.0,0.122991,0.709250,-1.0,-0.572683,0.318270,0.768330,1.000000
hour_cos,100000.0,-0.161457,0.675114,-1.0,-0.769539,-0.284503,0.462152,1.000000
weekday_sin,100000.0,0.076037,0.729768,-1.0,-0.709117,0.133878,0.851553,1.000000
weekday_cos,100000.0,-0.044026,0.678033,-1.0,-0.670652,-0.082777,0.648834,1.000000
is_weekend,100000.0,0.170980,0.376493,0.0,0.000000,0.000000,0.000000,1.000000
source_port_value,100000.0,46256.127710,13881.623215,1025.0,38732.000000,49537.000000,57193.250000,65534.000000
is_privileged_port,100000.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
is_registered_port,100000.0,0.466330,0.498868,0.0,0.000000,0.000000,1.000000,1.000000
is_dynamic_port,100000.0,0.533670,0.498868,0.0,0.000000,1.000000,1.000000,1.000000
language_present,100000.0,0.439530,0.496332,0.0,0.000000,0.000000,1.000000,1.000000


## Full feature-sidecar export

Features are exported as a separate sidecar containing event identity and the 21 primary features.

The export:

- processes every record in source-arrival order;
- preserves client state across chunks;
- verifies label alignment;
- writes atomically through partial files;
- records feature and source-code fingerprints;
- remains safe when the notebook is rerun.

In [9]:
from datetime import datetime, timezone
from itertools import zip_longest
import time

from src.features import (
    ALL_PRIMARY_FEATURES,
    DECAY_WINDOWS_SECONDS,
    FEATURE_VERSION,
    CausalClientFeatureTransformer,
    build_event_local_features,
)

FEATURES_DIRECTORY = (
    PROJECT_ROOT / "data" / "features"
)

FEATURES_OUTPUT = (
    FEATURES_DIRECTORY / "cj_primary_features.csv"
)

FEATURES_MANIFEST_OUTPUT = (
    FEATURES_DIRECTORY
    / "cj_primary_features_manifest.json"
)

FEATURE_CODE_PATH = PROJECT_ROOT / "src" / "features.py"

FEATURE_SIDECAR_COLUMNS = [
    "event_id",
    "record_hash",
    *ALL_PRIMARY_FEATURES,
]

In [10]:
def export_primary_features():
    FEATURES_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True,
    )

    partial_output = FEATURES_OUTPUT.with_name(
        FEATURES_OUTPUT.name + ".partial"
    )

    partial_manifest = FEATURES_MANIFEST_OUTPUT.with_name(
        FEATURES_MANIFEST_OUTPUT.name + ".partial"
    )

    if (
        FEATURES_OUTPUT.exists()
        or FEATURES_MANIFEST_OUTPUT.exists()
    ):
        raise FileExistsError(
            "Final feature artifacts already exist."
        )

    remaining_partials = [
        path
        for path in [
            partial_output,
            partial_manifest,
        ]
        if path.exists()
    ]

    if remaining_partials:
        raise FileExistsError(
            f"Previous incomplete export exists: "
            f"{remaining_partials}"
        )

    cleaned_reader = pd.read_csv(
        CLEANED_PATH,
        usecols=[
            "event_id",
            "record_hash",
            "client_ip_canonical",
            "timestamp_normalized",
            "source_port_normalized",
            "language",
        ],
        dtype={
            "event_id": str,
            "record_hash": str,
            "client_ip_canonical": str,
        },
        keep_default_na=False,
        chunksize=CHUNK_SIZE,
    )

    label_reader = pd.read_csv(
        LABELS_PATH,
        usecols=[
            "event_id",
            "record_hash",
        ],
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE,
    )

    transformer = CausalClientFeatureTransformer()

    feature_minimum = None
    feature_maximum = None
    feature_sum = None
    feature_nonzero_count = None

    written_rows = 0
    chunk_count = 0
    timestamp_reversals = 0

    missing_chunk = object()
    started_at = time.perf_counter()

    for chunk_number, (
        cleaned_chunk,
        label_chunk,
    ) in enumerate(
        zip_longest(
            cleaned_reader,
            label_reader,
            fillvalue=missing_chunk,
        ),
        start=1,
    ):
        if (
            cleaned_chunk is missing_chunk
            or label_chunk is missing_chunk
        ):
            raise RuntimeError(
                "Cleaned and label files have "
                "different chunk counts."
            )

        if len(cleaned_chunk) != len(label_chunk):
            raise RuntimeError(
                f"Row-count mismatch in chunk "
                f"{chunk_number}."
            )

        cleaned_identity = cleaned_chunk[
            ["event_id", "record_hash"]
        ].reset_index(drop=True)

        label_identity = label_chunk[
            ["event_id", "record_hash"]
        ].reset_index(drop=True)

        if not cleaned_identity.equals(label_identity):
            raise RuntimeError(
                f"Identity mismatch in chunk "
                f"{chunk_number}."
            )

        event_local = build_event_local_features(
            cleaned_chunk
        ).reset_index(drop=True)

        past_only = transformer.transform_chunk(
            cleaned_chunk
        ).reset_index(drop=True)

        feature_matrix = pd.concat(
            [
                event_local,
                past_only,
            ],
            axis=1,
        )

        if (
            list(feature_matrix.columns)
            != ALL_PRIMARY_FEATURES
        ):
            raise RuntimeError(
                "Generated features violate the contract."
            )

        feature_values = feature_matrix.to_numpy(
            dtype=np.float64
        )

        if not np.isfinite(feature_values).all():
            raise RuntimeError(
                f"Non-finite feature found in chunk "
                f"{chunk_number}."
            )

        chunk_minimum = feature_values.min(axis=0)
        chunk_maximum = feature_values.max(axis=0)
        chunk_sum = feature_values.sum(axis=0)
        chunk_nonzero = (
            feature_values != 0
        ).sum(axis=0)

        if feature_minimum is None:
            feature_minimum = chunk_minimum
            feature_maximum = chunk_maximum
            feature_sum = chunk_sum
            feature_nonzero_count = chunk_nonzero
        else:
            feature_minimum = np.minimum(
                feature_minimum,
                chunk_minimum,
            )

            feature_maximum = np.maximum(
                feature_maximum,
                chunk_maximum,
            )

            feature_sum += chunk_sum
            feature_nonzero_count += chunk_nonzero

        timestamp_reversals += int(
            feature_matrix[
                "timestamp_reversal_flag"
            ].sum()
        )

        output_chunk = pd.concat(
            [
                cleaned_identity,
                feature_matrix,
            ],
            axis=1,
        )

        if list(output_chunk.columns) != FEATURE_SIDECAR_COLUMNS:
            raise RuntimeError(
                "Feature sidecar violates its schema."
            )

        output_chunk.to_csv(
            partial_output,
            mode="w" if chunk_number == 1 else "a",
            header=(chunk_number == 1),
            index=False,
            na_rep=NULL_SENTINEL,
            float_format="%.9g",
            lineterminator="\n",
        )

        written_rows += len(output_chunk)
        chunk_count += 1

    expected_rows = cleaning_manifest["output_rows"]

    if written_rows != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows:,} rows but "
            f"generated {written_rows:,}."
        )

    elapsed_seconds = (
        time.perf_counter() - started_at
    )

    feature_statistics = {}

    for position, feature_name in enumerate(
        ALL_PRIMARY_FEATURES
    ):
        feature_statistics[feature_name] = {
            "minimum": float(
                feature_minimum[position]
            ),
            "maximum": float(
                feature_maximum[position]
            ),
            "mean": float(
                feature_sum[position] / written_rows
            ),
            "nonzero_count": int(
                feature_nonzero_count[position]
            ),
        }

    constant_features = [
        feature_name
        for position, feature_name in enumerate(
            ALL_PRIMARY_FEATURES
        )
        if (
            feature_minimum[position]
            == feature_maximum[position]
        )
    ]

    output_hash = calculate_file_sha256(
        partial_output
    )

    feature_code_hash = calculate_file_sha256(
        FEATURE_CODE_PATH
    )

    manifest = {
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "feature_version": FEATURE_VERSION,
        "input_file": CLEANED_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "label_file": LABELS_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "output_file": FEATURES_OUTPUT.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "input_sha256": actual_cleaned_hash,
        "label_sha256": actual_labels_hash,
        "output_sha256": output_hash,
        "feature_code_file": FEATURE_CODE_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "feature_code_sha256": feature_code_hash,
        "input_rows": expected_rows,
        "output_rows": written_rows,
        "chunk_count": chunk_count,
        "chunk_size": CHUNK_SIZE,
        "feature_columns": ALL_PRIMARY_FEATURES,
        "sidecar_columns": FEATURE_SIDECAR_COLUMNS,
        "feature_count": len(ALL_PRIMARY_FEATURES),
        "feature_registry": feature_registry.to_dict(
            orient="records"
        ),
        "ordering_policy": "source_arrival_order",
        "state_update_policy": "transform_then_update",
        "history_policy": "past_events_only",
        "timestamp_reversal_policy": (
            "flag_and_clip_negative_gap_to_zero"
        ),
        "decay_windows_seconds": list(
            DECAY_WINDOWS_SECONDS
        ),
        "timestamp_reversal_count": (
            timestamp_reversals
        ),
        "state_summary": transformer.state_summary(),
        "constant_features": constant_features,
        "feature_statistics": feature_statistics,
        "elapsed_seconds": round(
            elapsed_seconds,
            2,
        ),
    }

    with partial_manifest.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            manifest,
            file,
            indent=2,
        )

    partial_output.replace(FEATURES_OUTPUT)
    partial_manifest.replace(
        FEATURES_MANIFEST_OUTPUT
    )

    return manifest

In [14]:
def load_or_export_primary_features():
    output_exists = FEATURES_OUTPUT.exists()
    manifest_exists = (
        FEATURES_MANIFEST_OUTPUT.exists()
    )

    if output_exists != manifest_exists:
        raise RuntimeError(
            "Incomplete feature artifact set: CSV and "
            "manifest must both exist or both be absent."
        )

    if not output_exists:
        print("Creating primary feature sidecar...")
        return export_primary_features()

    with FEATURES_MANIFEST_OUTPUT.open(
        "r",
        encoding="utf-8",
    ) as file:
        existing_manifest = json.load(file)

    if (
        calculate_file_sha256(CLEANED_PATH)
        != existing_manifest["input_sha256"]
    ):
        raise RuntimeError(
            "Cleaned input changed after feature export."
        )

    if (
        calculate_file_sha256(LABELS_PATH)
        != existing_manifest["label_sha256"]
    ):
        raise RuntimeError(
            "Weak-label sidecar changed after feature export."
        )

    if (
        calculate_file_sha256(FEATURES_OUTPUT)
        != existing_manifest["output_sha256"]
    ):
        raise RuntimeError(
            "Feature-sidecar fingerprint failed."
        )

    if (
        calculate_file_sha256(FEATURE_CODE_PATH)
        != existing_manifest["feature_code_sha256"]
    ):
        raise RuntimeError(
            "Feature code changed after export. "
            "A new feature version is required."
        )

    if (
        existing_manifest["feature_version"]
        != FEATURE_VERSION
    ):
        raise RuntimeError(
            "Feature version mismatch."
        )

    if (
        existing_manifest["feature_columns"]
        != ALL_PRIMARY_FEATURES
    ):
        raise RuntimeError(
            "Manifest feature contract differs from code."
        )

    print("Reusing verified primary feature sidecar.")
    return existing_manifest


feature_export_result = (
    load_or_export_primary_features()
)

feature_export_result

Reusing verified primary feature sidecar.


{'created_at_utc': '2026-09-01T09:22:39.669160+00:00',
 'feature_version': 'feature-v1',
 'input_file': 'data/processed/cj_cleaned.csv',
 'label_file': 'data/labels/cj_weak_labels.csv',
 'output_file': 'data/features/cj_primary_features.csv',
 'input_sha256': 'c022cee1cb2de32f7bb1db1d38d5d1dad1079b2fb590f854f9aead7fa54995f2',
 'label_sha256': '17e6c785c07056fd973957192653c0fb97706477477c2ac009c9c5691857e444',
 'output_sha256': '0dca75a12f5fe290dcc3e398951b3e9dd1ecc823f380a64024dfc39801ad6b8e',
 'feature_code_file': 'src/features.py',
 'feature_code_sha256': 'afd0fa4ae6bcd00d0dfa4be4051c49b8d90319c114002e27147b0a3a69faa8b8',
 'input_rows': 2062361,
 'output_rows': 2062361,
 'chunk_count': 21,
 'chunk_size': 100000,
 'feature_columns': ['hour_sin',
  'hour_cos',
  'weekday_sin',
  'weekday_cos',
  'is_weekend',
  'source_port_value',
  'is_privileged_port',
  'is_registered_port',
  'is_dynamic_port',
  'language_present',
  'language_length',
  'client_prior_event_count_log1p',
  'has_p

## Independent feature-sidecar verification

The physical feature file is checked independently of the exporter. This verifies identities, schema, numeric validity, statistics, fingerprints, and manifest reconciliation after CSV serialization.

In [12]:
feature_manifest = feature_export_result

assert (
    calculate_file_sha256(FEATURES_OUTPUT)
    == feature_manifest["output_sha256"]
), "Feature fingerprint mismatch."

assert (
    calculate_file_sha256(FEATURE_CODE_PATH)
    == feature_manifest["feature_code_sha256"]
), "Feature-code fingerprint mismatch."

feature_header = pd.read_csv(
    FEATURES_OUTPUT,
    nrows=0,
).columns.tolist()

assert feature_header == FEATURE_SIDECAR_COLUMNS

assert set(ALL_PRIMARY_FEATURES).isdisjoint(
    RESTRICTED_LABEL_SOURCES
    | TARGET_METADATA_COLUMNS
    | GROUPING_ONLY_COLUMNS
)

cleaned_identity_reader = pd.read_csv(
    CLEANED_PATH,
    usecols=["event_id", "record_hash"],
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

label_identity_reader = pd.read_csv(
    LABELS_PATH,
    usecols=["event_id", "record_hash"],
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

feature_reader = pd.read_csv(
    FEATURES_OUTPUT,
    dtype={
        "event_id": str,
        "record_hash": str,
    },
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

verified_rows = 0
verified_chunks = 0
verified_reversals = 0

verified_minimum = None
verified_maximum = None
verified_sum = None
verified_nonzero = None

missing_chunk = object()

for chunk_number, chunks in enumerate(
    zip_longest(
        cleaned_identity_reader,
        label_identity_reader,
        feature_reader,
        fillvalue=missing_chunk,
    ),
    start=1,
):
    cleaned_identity, label_identity, feature_chunk = chunks

    if any(
        chunk is missing_chunk
        for chunk in chunks
    ):
        raise RuntimeError(
            "Input and feature files have different "
            "chunk counts."
        )

    if not (
        len(cleaned_identity)
        == len(label_identity)
        == len(feature_chunk)
    ):
        raise RuntimeError(
            f"Row-count mismatch in chunk {chunk_number}."
        )

    feature_identity = feature_chunk[
        ["event_id", "record_hash"]
    ]

    cleaned_identity = cleaned_identity.reset_index(
        drop=True
    )

    label_identity = label_identity.reset_index(
        drop=True
    )

    feature_identity = feature_identity.reset_index(
        drop=True
    )

    if not cleaned_identity.equals(label_identity):
        raise RuntimeError(
            f"Cleaned-label identity mismatch in "
            f"chunk {chunk_number}."
        )

    if not cleaned_identity.equals(feature_identity):
        raise RuntimeError(
            f"Cleaned-feature identity mismatch in "
            f"chunk {chunk_number}."
        )

    feature_values = feature_chunk[
        ALL_PRIMARY_FEATURES
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(feature_values).all():
        raise RuntimeError(
            f"Non-finite value in feature chunk "
            f"{chunk_number}."
        )

    chunk_minimum = feature_values.min(axis=0)
    chunk_maximum = feature_values.max(axis=0)
    chunk_sum = feature_values.sum(axis=0)
    chunk_nonzero = (
        feature_values != 0
    ).sum(axis=0)

    if verified_minimum is None:
        verified_minimum = chunk_minimum
        verified_maximum = chunk_maximum
        verified_sum = chunk_sum
        verified_nonzero = chunk_nonzero
    else:
        verified_minimum = np.minimum(
            verified_minimum,
            chunk_minimum,
        )

        verified_maximum = np.maximum(
            verified_maximum,
            chunk_maximum,
        )

        verified_sum += chunk_sum
        verified_nonzero += chunk_nonzero

    verified_reversals += int(
        feature_chunk[
            "timestamp_reversal_flag"
        ].sum()
    )

    verified_rows += len(feature_chunk)
    verified_chunks += 1

assert verified_rows == feature_manifest["output_rows"]
assert verified_chunks == feature_manifest["chunk_count"]

verified_constant_features = []

for position, feature_name in enumerate(
    ALL_PRIMARY_FEATURES
):
    recorded = feature_manifest[
        "feature_statistics"
    ][feature_name]

    assert np.isclose(
        verified_minimum[position],
        recorded["minimum"],
        rtol=1e-6,
        atol=1e-7,
    )

    assert np.isclose(
        verified_maximum[position],
        recorded["maximum"],
        rtol=1e-6,
        atol=1e-7,
    )

    verified_mean = (
        verified_sum[position] / verified_rows
    )

    assert np.isclose(
        verified_mean,
        recorded["mean"],
        rtol=1e-6,
        atol=1e-7,
    )

    assert (
        int(verified_nonzero[position])
        == recorded["nonzero_count"]
    )

    if (
        verified_minimum[position]
        == verified_maximum[position]
    ):
        verified_constant_features.append(
            feature_name
        )

assert (
    verified_constant_features
    == feature_manifest["constant_features"]
)

assert (
    verified_reversals
    == feature_manifest["timestamp_reversal_count"]
)

partial_files = [
    FEATURES_OUTPUT.with_name(
        FEATURES_OUTPUT.name + ".partial"
    ),
    FEATURES_MANIFEST_OUTPUT.with_name(
        FEATURES_MANIFEST_OUTPUT.name + ".partial"
    ),
]

assert not any(
    path.exists()
    for path in partial_files
)

print("Verified rows:", f"{verified_rows:,}")
print("Verified chunks:", verified_chunks)
print("Identity alignment: passed")
print("Serialized schema: passed")
print("Finite numeric features: passed")
print("Feature statistics: passed")
print("Output fingerprint: passed")
print(
    "Timestamp reversals:",
    verified_reversals,
)
print(
    "Constant features:",
    verified_constant_features,
)
print(
    "Final client state:",
    feature_manifest["state_summary"],
)
print(
    "Feature file size (MiB):",
    round(
        FEATURES_OUTPUT.stat().st_size
        / (1024 ** 2),
        2,
    ),
)
print("Partial files remaining: 0")

Verified rows: 2,062,361
Verified chunks: 21
Identity alignment: passed
Serialized schema: passed
Finite numeric features: passed
Feature statistics: passed
Output fingerprint: passed
Timestamp reversals: 585
Constant features: ['is_privileged_port']
Final client state: {'client_count': 16680, 'processed_events': 2062361, 'client_port_pairs': 191556}
Feature file size (MiB): 371.61
Partial files remaining: 0


In [13]:
constant_features = set(
    feature_export_result["constant_features"]
)

model_feature_registry = feature_registry.copy()

model_feature_registry["model_status"] = np.where(
    model_feature_registry["feature"].isin(
        constant_features
    ),
    "exclude_zero_variance",
    "eligible",
)

MODEL_ELIGIBLE_FEATURES = (
    model_feature_registry.loc[
        model_feature_registry["model_status"]
        == "eligible",
        "feature",
    ].tolist()
)

assert len(MODEL_ELIGIBLE_FEATURES) == 20

assert set(MODEL_ELIGIBLE_FEATURES).isdisjoint(
    RESTRICTED_LABEL_SOURCES
    | TARGET_METADATA_COLUMNS
    | GROUPING_ONLY_COLUMNS
)

print("Engineered features:", len(ALL_PRIMARY_FEATURES))
print("Model-eligible features:", len(MODEL_ELIGIBLE_FEATURES))
print("Zero-variance exclusions:", sorted(constant_features))

model_feature_registry

Engineered features: 21
Model-eligible features: 20
Zero-variance exclusions: ['is_privileged_port']


,feature,feature_family,source,model_status
0,hour_sin,event_local,timestamp_normalized,eligible
1,hour_cos,event_local,timestamp_normalized,eligible
2,weekday_sin,event_local,timestamp_normalized,eligible
3,weekday_cos,event_local,timestamp_normalized,eligible
4,is_weekend,event_local,timestamp_normalized,eligible
5,source_port_value,event_local,source_port_normalized,eligible
6,is_privileged_port,event_local,source_port_normalized,exclude_zero_variance
7,is_registered_port,event_local,source_port_normalized,eligible
8,is_dynamic_port,event_local,source_port_normalized,eligible
9,language_present,event_local,language,eligible


## Conclusion

Practical 5 produced a verified feature sidecar for all 2,062,361 events.

### Feature design

- 11 event-local features
- 10 past-only behavioural features
- 21 engineered features in total
- 20 model-eligible features
- 1 zero-variance feature excluded from modeling

The feature transformer maintained independent online state for 16,680 clients and observed 191,556 client–port relationships. It detected and preserved 585 timestamp reversals.

The current event was transformed before updating client state, ensuring that features use only previously observed events. Chunk-boundary invariance and future independence were verified synthetically.

Columns used to generate weak labels, direct client identifiers, provenance fields, and target metadata were excluded from the primary feature matrix. This prevents direct rule reproduction, identity memorization, and target leakage.

`is_privileged_port` was retained in the immutable feature artifact for schema consistency but marked as model-ineligible because its full-dataset variance is zero.